# Task 3 — Final Gender SAM25 refit

Train **one fresh model on all 32,773 development images**. Keep the frozen SAM25
recipe and corrected Gender labels. Save epoch **25**, with cosine **T_max=30**.

Use a fresh **Colab L4**. The new code must be on GitHub before Run All.
Reuse the existing teacher data ZIP in Drive. Results go to a separate refit folder.
This notebook does not score holdout/test images or replace the existing five-model artifact.

## 1. Mount Drive and select the repository

In [1]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
CHECKOUT_DIR = Path("/content/MLA2")
DRIVE_PROJECT = Path("/content/drive/MyDrive/MLA2")
DATA_ZIP = DRIVE_PROJECT / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
drive.mount("/content/drive", force_remount=False)

Mounted at /content/drive


## 2. Fetch code from GitHub

Keep local edits safe: repository updates use a fast-forward merge.

In [2]:
def run_checked(command):
    return subprocess.run([str(x) for x in command], check=True)


if (CHECKOUT_DIR / ".git").is_dir():
    remote = subprocess.check_output(
        ["git", "-C", str(CHECKOUT_DIR), "remote", "get-url", "origin"], text=True
    ).strip()
    if remote != REPO_URL:
        raise RuntimeError("The local checkout belongs to another repository")
    run_checked(["git", "-C", CHECKOUT_DIR, "fetch", "origin", BRANCH])
    run_checked(["git", "-C", CHECKOUT_DIR, "switch", BRANCH])
    run_checked(["git", "-C", CHECKOUT_DIR, "merge", "--ff-only", f"origin/{BRANCH}"])
else:
    run_checked(["git", "clone", "--branch", BRANCH, REPO_URL, CHECKOUT_DIR])
REPO_DIR = CHECKOUT_DIR / "core" if (CHECKOUT_DIR / "core/src/fashion").is_dir() else CHECKOUT_DIR
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
print("Code commit:")
run_checked(["git", "rev-parse", "HEAD"])

Code commit:


CompletedProcess(args=['git', 'rev-parse', 'HEAD'], returncode=0)

## 3. Reuse the existing teacher images

Read the canonical development rows. Extract only missing training images.
The archive cannot overwrite Git code or the saved split. The refit preflight
then checks every development image against its recorded hash.

In [3]:
import pandas as pd

splits = pd.read_csv(REPO_DIR / "data/processed/splits.csv", keep_default_na=False)
paths = splits.loc[splits.partition.eq("development"), "path"].tolist()
missing = [source_path for source_path in paths if not (REPO_DIR / source_path).is_file()]
if missing:
    local_zip = Path("/content/task3-refit-data.zip")
    shutil.copyfile(DATA_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as archive:
        for source_path in missing:
            if not source_path.startswith("data/raw/teacher/train/images_train/"):
                raise ValueError(f"Unexpected development image path: {source_path}")
            target = (REPO_DIR / source_path).resolve()
            if not target.is_relative_to(REPO_DIR.resolve()):
                raise ValueError("Image path leaves the repository")
            target.parent.mkdir(parents=True, exist_ok=True)
            partial = target.with_suffix(target.suffix + ".partial")
            with archive.open(source_path) as source, partial.open("wb") as output:
                shutil.copyfileobj(source, output)
            partial.replace(target)
print(f"Development images ready: {len(paths):,}")

Development images ready: 32,773


## 4. Review the fixed recipe

SmallCNN + GeM p=3, dropout **0.30**, MixUp **0.2**, SAM **0.05**.
AdamW: learning rate **0.001**, weight decay **0.0001**, batch **128**, seed **2753**.
Keep translation ±2 px (50%), mild darkening (25%) and grayscale (10%).
Normalize from all development images. No validation split, early stopping or model selection.

In [4]:
import json

import torch

from fashion.train.task3_gender_sam25_refit import EXPERIMENT, SOURCE_CONFIG

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before training")
saved = json.loads((REPO_DIR / SOURCE_CONFIG).read_text())
print("GPU:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__)
print("Epochs:", saved["epochs"], "Cosine T_max:", saved["cosine_t_max"])
print("Class labels: Boys, Girls, Men, Unisex, Women")
print("Output:", DRIVE_TASK_DIR / "experiments" / EXPERIMENT / "gender")

GPU: NVIDIA L4 PyTorch: 2.11.0+cu128
Epochs: 25 Cosine T_max: 30
Class labels: Boys, Girls, Men, Unisex, Women
Output: /content/drive/MyDrive/MLA2/task3/experiments/t3_gender_name_truth_mixup_alpha020_sam005_epoch25_refit/gender


## 5. Train and save the final refit

Run the full preflight, register the run, then train from scratch for 25 epochs.
Every epoch records its mixed training loss and verifies that each development row was used once.
Repeat Run All verifies and reuses a completed refit; an interrupted fit restarts from scratch.

In [5]:
from fashion.train.task3_gender_sam25_refit import run_gender_sam25_refit

result = run_gender_sam25_refit(
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,),
)
print("Status:", result["status"])
print("Reused:", result["reused"])
print("Model manifest:", result["manifest_path"])

Checking 32,773 development image hashes
Epoch 1/25: mixed training loss 0.7382
Epoch 2/25: mixed training loss 0.6226
Epoch 3/25: mixed training loss 0.5904
Epoch 4/25: mixed training loss 0.5471
Epoch 5/25: mixed training loss 0.5378
Epoch 6/25: mixed training loss 0.5130
Epoch 7/25: mixed training loss 0.5101
Epoch 8/25: mixed training loss 0.4914
Epoch 9/25: mixed training loss 0.4930
Epoch 10/25: mixed training loss 0.4621
Epoch 11/25: mixed training loss 0.4722
Epoch 12/25: mixed training loss 0.4506
Epoch 13/25: mixed training loss 0.4342
Epoch 14/25: mixed training loss 0.4354
Epoch 15/25: mixed training loss 0.4316
Epoch 16/25: mixed training loss 0.4341
Epoch 17/25: mixed training loss 0.4347
Epoch 18/25: mixed training loss 0.4294
Epoch 19/25: mixed training loss 0.4201
Epoch 20/25: mixed training loss 0.3917
Epoch 21/25: mixed training loss 0.4145
Epoch 22/25: mixed training loss 0.3879
Epoch 23/25: mixed training loss 0.4149
Epoch 24/25: mixed training loss 0.3662
Epoch 25

## 6. Check the saved training record

These losses come from mixed training images. They are not validation scores.
The saved checkpoint and normalization form one model for later inference.
No holdout/test predictions or submission file are created by this run.

In [6]:
manifest_dir = Path(result["manifest_path"]).parent
history_path = manifest_dir / result["files"]["history.csv"]["path"]
history = pd.read_csv(history_path)
assert history.epoch.tolist() == list(range(1, 26))
assert history.training_rows.eq(32773).all()
assert history.selected_checkpoint.tolist() == [False] * 24 + [True]
display(history.tail())
print("Checkpoint:", manifest_dir / result["files"]["final_epoch.pt"]["path"])
print("Normalization:", manifest_dir / result["files"]["normalization.json"]["path"])

,epoch,learning_rate,train_mixed_loss,sam_perturbed_loss,training_rows,optimizer_steps,selected_checkpoint
20,21,0.000258,0.414543,0.446142,32773,257,False
21,22,0.000214,0.387910,0.420547,32773,257,False
22,23,0.000174,0.414876,0.446319,32773,257,False
23,24,0.000137,0.366245,0.396772,32773,257,False
24,25,0.000105,0.387814,0.417654,32773,257,True


Checkpoint: /content/drive/MyDrive/MLA2/task3/experiments/t3_gender_name_truth_mixup_alpha020_sam005_epoch25_refit/gender/t3_gender_name_truth_mixup_alpha020_sam005_epoch25_refit_20260907T061926Z_db0fc1ee/final_epoch.pt
Normalization: /content/drive/MyDrive/MLA2/task3/experiments/t3_gender_name_truth_mixup_alpha020_sam005_epoch25_refit/gender/t3_gender_name_truth_mixup_alpha020_sam005_epoch25_refit_20260907T061926Z_db0fc1ee/normalization.json
